# Neural Network Comparison for Multi-Route Model

This notebook tests whether using a dense neural network (NN) architecture improves upon the existing
Ridge, Random Forest, and XGBoost models for flight delay prediction.

Note that the neural network is not expected to significantly outperform existing models due to a few factors:
- Small dataset (~7,200 training samples), which limits the advantage of NN
- One feature (```delay_rate_lag1```) is by far the most dominant (65% relative importance)
- Using another non-linear model, Random Forest, results in a comparable performance

**NN Architecture:**
- 2 hidden layers: keeping the network shallow, following the three reasons above
- layer sizes 32 -> 16 neurons: first hidden layer size of 32 is to roughly match the 39 input parameters
- ReLU activation
- Dropout 0.3: to avoid overfitting the small dataset
- Early stopping on validation loss (patience=20)
- Adam optimiser: learning rate 0.001, batch size 64

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TF = True
    tf.random.set_seed(42)
    print(f"TensorFlow {tf.__version__} available")
except ImportError:
    HAS_TF = False
    print("TensorFlow not installed. Neural network sections will be skipped.")

np.random.seed(42)

%matplotlib inline

## 1. Data Preparation

In [ ]:
# Load multi-route data
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

## 2. Filter Low-Volume Airline-Routes

In [ ]:
volume_threshold = 50
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean().reset_index()
airline_route_volume.columns = ['airline_route', 'avg_volume']
high_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] >= volume_threshold]['airline_route'].tolist()

excluded = airline_route_volume[airline_route_volume['avg_volume'] < volume_threshold].sort_values('avg_volume')
print(f"Volume threshold: {volume_threshold} flights/month")
print(f"Excluded airline-routes: {len(excluded)}")

df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()
print(f"\nRecords before filtering: {len(df)}")
print(f"Records after filtering:  {len(df_filtered)}")

In [ ]:
# Exclude Melbourne-Hobart route (anomalous 2019 data, see notebook 6c)
anomalous_routes = ['Melbourne_Hobart', 'Adelaide_Sydney']
records_before = len(df_filtered)
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()
print(f"Excluded anomalous routes: {anomalous_routes}")
print(f"Records before: {records_before}")
print(f"Records after:  {len(df_filtered)}")

## 3. Feature Engineering

In [ ]:
# Lagged features
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Weather features with transformations
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / df_filtered['rainy_days_arr'].max())
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / df_filtered['temp_volatility_total'].max())
df_filtered['extreme_weather_days_total'] = df_filtered['extreme_weather_days_dep'] + df_filtered['extreme_weather_days_arr']

# Drop NaN (from lag features)
df_clean = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
print(f"Rows after dropping NaN: {len(df_clean)}")

## 4. Train/Validation/Test Split

In [ ]:
train_mask = (((df_clean['year'] >= 2010) & (df_clean['year'] <= 2017)) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2023): {train_mask.sum()} samples")
print(f"Validation (2018, 2024): {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")

In [ ]:
# Define final feature set
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']
weather_features = ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp', 'extreme_weather_days_total']
holiday_features = ['n_public_holidays_total', 'pct_school_holiday']

FINAL_FEATURES = base_features + weather_features + holiday_features

print(f"Total features: {len(FINAL_FEATURES)}")
print(f"\nFeature breakdown:")
print(f"  - Airline encoding: {len(airline_cols)}")
print(f"  - Route encoding:   {len(route_cols)}")
print(f"  - Month encoding:   2 (sin, cos)")
print(f"  - Lag features:     2 (lag1, gradient)")
print(f"  - Volume:           1 (sectors_scheduled)")
print(f"  - Weather:          3 (rainy, temp_volatility, extreme)")
print(f"  - Holidays:         2 (public, school)")

In [ ]:
# Prepare data
X_train = df_clean.loc[train_mask, FINAL_FEATURES].values
X_val = df_clean.loc[val_mask, FINAL_FEATURES].values
X_test = df_clean.loc[test_mask, FINAL_FEATURES].values

y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Data prepared.")
print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

## 5. Existing Model Baselines

In [ ]:
reg_results = []
clf_results = []

# --- Regression ---
print("Training Ridge Regression...")
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train_reg)
ridge_val_pred = ridge.predict(X_val_scaled)
ridge_test_pred = ridge.predict(X_test_scaled)
ridge_val_r2 = r2_score(y_val_reg, ridge_val_pred)
ridge_test_r2 = r2_score(y_test_reg, ridge_test_pred)
ridge_test_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_test_pred))
ridge_test_mae = mean_absolute_error(y_test_reg, ridge_test_pred)
reg_results.append({'Model': 'Ridge', 'Val_R2': ridge_val_r2, 'Test_R2': ridge_test_r2,
                    'Test_RMSE': ridge_test_rmse, 'Test_MAE': ridge_test_mae})
print(f"  Val R²: {ridge_val_r2:.4f}, Test R²: {ridge_test_r2:.4f}")

print("Training Random Forest Regression...")
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train_reg)
rf_val_pred = rf_reg.predict(X_val)
rf_test_pred = rf_reg.predict(X_test)
rf_val_r2 = r2_score(y_val_reg, rf_val_pred)
rf_test_r2 = r2_score(y_test_reg, rf_test_pred)
rf_test_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_test_pred))
rf_test_mae = mean_absolute_error(y_test_reg, rf_test_pred)
reg_results.append({'Model': 'Random Forest', 'Val_R2': rf_val_r2, 'Test_R2': rf_test_r2,
                    'Test_RMSE': rf_test_rmse, 'Test_MAE': rf_test_mae})
print(f"  Val R²: {rf_val_r2:.4f}, Test R²: {rf_test_r2:.4f}")

In [ ]:
# --- Classification ---
print("Training Logistic Regression...")
logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train_clf)
logreg_test_pred = logreg.predict(X_test_scaled)
logreg_test_proba = logreg.predict_proba(X_test_scaled)[:, 1]
logreg_val_f1 = f1_score(y_val_clf, logreg.predict(X_val_scaled))
logreg_test_f1 = f1_score(y_test_clf, logreg_test_pred)
logreg_test_auc = roc_auc_score(y_test_clf, logreg_test_proba)
logreg_test_precision = precision_score(y_test_clf, logreg_test_pred)
logreg_test_recall = recall_score(y_test_clf, logreg_test_pred)
clf_results.append({'Model': 'Logistic', 'Val_F1': logreg_val_f1, 'Test_F1': logreg_test_f1,
                    'Test_AUC': logreg_test_auc, 'Test_Precision': logreg_test_precision,
                    'Test_Recall': logreg_test_recall})
print(f"  Val F1: {logreg_val_f1:.4f}, Test F1: {logreg_test_f1:.4f}")

print("Training Random Forest Classification...")
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train_clf)
rf_clf_test_pred = rf_clf.predict(X_test)
rf_clf_test_proba = rf_clf.predict_proba(X_test)[:, 1]
rf_clf_val_f1 = f1_score(y_val_clf, rf_clf.predict(X_val))
rf_clf_test_f1 = f1_score(y_test_clf, rf_clf_test_pred)
rf_clf_test_auc = roc_auc_score(y_test_clf, rf_clf_test_proba)
rf_clf_test_precision = precision_score(y_test_clf, rf_clf_test_pred)
rf_clf_test_recall = recall_score(y_test_clf, rf_clf_test_pred)
clf_results.append({'Model': 'Random Forest', 'Val_F1': rf_clf_val_f1, 'Test_F1': rf_clf_test_f1,
                    'Test_AUC': rf_clf_test_auc, 'Test_Precision': rf_clf_test_precision,
                    'Test_Recall': rf_clf_test_recall})
print(f"  Val F1: {rf_clf_val_f1:.4f}, Test F1: {rf_clf_test_f1:.4f}")

if HAS_XGB:
    print("Training XGBoost Classification...")
    xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                 min_child_weight=5, random_state=42, n_jobs=-1)
    xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
    xgb_test_pred = xgb_clf.predict(X_test)
    xgb_test_proba = xgb_clf.predict_proba(X_test)[:, 1]
    xgb_val_f1 = f1_score(y_val_clf, xgb_clf.predict(X_val))
    xgb_test_f1 = f1_score(y_test_clf, xgb_test_pred)
    xgb_test_auc = roc_auc_score(y_test_clf, xgb_test_proba)
    xgb_test_precision = precision_score(y_test_clf, xgb_test_pred)
    xgb_test_recall = recall_score(y_test_clf, xgb_test_pred)
    clf_results.append({'Model': 'XGBoost', 'Val_F1': xgb_val_f1, 'Test_F1': xgb_test_f1,
                        'Test_AUC': xgb_test_auc, 'Test_Precision': xgb_test_precision,
                        'Test_Recall': xgb_test_recall})
    print(f"  Val F1: {xgb_val_f1:.4f}, Test F1: {xgb_test_f1:.4f}")

print("\nBaseline models trained.")

## 6. Neural Network — Regression

In [ ]:
def build_nn(input_dim, hidden1=32, hidden2=16, dropout=0.3, output_activation='linear'):
    """Build a feedforward neural network.
    
    Architecture: Input -> Dense(hidden1, ReLU) -> Dropout -> Dense(hidden2, ReLU) -> Dropout -> Dense(1)
    """
    model = keras.Sequential([
        layers.Dense(hidden1, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(dropout),
        layers.Dense(hidden2, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(1, activation=output_activation)
    ])
    return model

print("Neural network builder defined.")
print(f"\nArchitecture: Input({X_train_scaled.shape[1]}) \u2192 Dense(32, ReLU) \u2192 Dropout(0.3) \u2192 Dense(16, ReLU) \u2192 Dropout(0.3) \u2192 Dense(1)")

In [ ]:
# Build and compile regression NN
tf.random.set_seed(42)
np.random.seed(42)

nn_reg = build_nn(input_dim=X_train_scaled.shape[1], hidden1=32, hidden2=16, dropout=0.3,
                  output_activation='linear')
nn_reg.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
               loss='mse', metrics=['mae'])

nn_reg.summary()
print(f"\nTotal parameters: {nn_reg.count_params():,}")
print(f"Training samples: {X_train_scaled.shape[0]:,}")
print(f"Samples-to-parameters ratio: {X_train_scaled.shape[0] / nn_reg.count_params():.1f}:1")

In [ ]:
# Train with early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=20, restore_best_weights=True, verbose=1
)

print("Training Neural Network (Regression)...")
history_reg = nn_reg.fit(
    X_train_scaled, y_train_reg,
    validation_data=(X_val_scaled, y_val_reg),
    epochs=500, batch_size=64,
    callbacks=[early_stop],
    verbose=0
)

best_epoch_reg = np.argmin(history_reg.history['val_loss']) + 1
print(f"Training complete. Best epoch: {best_epoch_reg} (of {len(history_reg.history['loss'])} total)")
print(f"Best val MSE: {min(history_reg.history['val_loss']):.6f}")

In [ ]:
# Training curve
fig, ax = plt.subplots(figsize=(10, 5))

epochs = range(1, len(history_reg.history['loss']) + 1)
ax.plot(epochs, history_reg.history['loss'], label='Train loss', color='steelblue')
ax.plot(epochs, history_reg.history['val_loss'], label='Validation loss', color='#e74c3c')
ax.axvline(best_epoch_reg, color='green', linestyle='--', alpha=0.7,
           label=f'Best epoch ({best_epoch_reg})')

ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Neural Network Regression: Training Curve')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate regression NN
nn_val_pred = nn_reg.predict(X_val_scaled, verbose=0).flatten()
nn_test_pred = nn_reg.predict(X_test_scaled, verbose=0).flatten()

nn_val_r2 = r2_score(y_val_reg, nn_val_pred)
nn_test_r2 = r2_score(y_test_reg, nn_test_pred)
nn_test_rmse = np.sqrt(mean_squared_error(y_test_reg, nn_test_pred))
nn_test_mae = mean_absolute_error(y_test_reg, nn_test_pred)

print(f"Neural Network Regression:")
print(f"  Val R²:    {nn_val_r2:.4f}")
print(f"  Test R²:   {nn_test_r2:.4f}")
print(f"  Test RMSE: {nn_test_rmse:.4f}")
print(f"  Test MAE:  {nn_test_mae:.4f}")

reg_results.append({'Model': 'Neural Network', 'Val_R2': nn_val_r2, 'Test_R2': nn_test_r2,
                    'Test_RMSE': nn_test_rmse, 'Test_MAE': nn_test_mae})

## 7. Neural Network — Classification

In [ ]:
# Build and compile classification NN
tf.random.set_seed(42)
np.random.seed(42)

nn_clf_model = build_nn(input_dim=X_train_scaled.shape[1], hidden1=32, hidden2=16, dropout=0.3,
                        output_activation='sigmoid')
nn_clf_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                     loss='binary_crossentropy', metrics=['accuracy'])

early_stop_clf = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=20, restore_best_weights=True, verbose=1
)

print("Training Neural Network (Classification)...")
history_clf = nn_clf_model.fit(
    X_train_scaled, y_train_clf.astype(np.float32),
    validation_data=(X_val_scaled, y_val_clf.astype(np.float32)),
    epochs=500, batch_size=64,
    callbacks=[early_stop_clf],
    verbose=0
)

best_epoch_clf = np.argmin(history_clf.history['val_loss']) + 1
print(f"Training complete. Best epoch: {best_epoch_clf} (of {len(history_clf.history['loss'])} total)")
print(f"Best val BCE: {min(history_clf.history['val_loss']):.6f}")

In [ ]:
# Training curve (classification)
fig, ax = plt.subplots(figsize=(10, 5))

epochs = range(1, len(history_clf.history['loss']) + 1)
ax.plot(epochs, history_clf.history['loss'], label='Train loss', color='steelblue')
ax.plot(epochs, history_clf.history['val_loss'], label='Validation loss', color='#e74c3c')
ax.axvline(best_epoch_clf, color='green', linestyle='--', alpha=0.7,
           label=f'Best epoch ({best_epoch_clf})')

ax.set_xlabel('Epoch')
ax.set_ylabel('Binary Cross-Entropy Loss')
ax.set_title('Neural Network Classification: Training Curve')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate classification NN
nn_clf_val_proba = nn_clf_model.predict(X_val_scaled, verbose=0).flatten()
nn_clf_test_proba = nn_clf_model.predict(X_test_scaled, verbose=0).flatten()
nn_clf_val_pred = (nn_clf_val_proba >= 0.5).astype(int)
nn_clf_test_pred = (nn_clf_test_proba >= 0.5).astype(int)

nn_clf_val_f1 = f1_score(y_val_clf, nn_clf_val_pred)
nn_clf_test_f1 = f1_score(y_test_clf, nn_clf_test_pred)
nn_clf_test_auc = roc_auc_score(y_test_clf, nn_clf_test_proba)
nn_clf_test_precision = precision_score(y_test_clf, nn_clf_test_pred)
nn_clf_test_recall = recall_score(y_test_clf, nn_clf_test_pred)

print(f"Neural Network Classification:")
print(f"  Val F1:       {nn_clf_val_f1:.4f}")
print(f"  Test F1:      {nn_clf_test_f1:.4f}")
print(f"  Test AUC:     {nn_clf_test_auc:.4f}")
print(f"  Test Precision: {nn_clf_test_precision:.4f}")
print(f"  Test Recall:    {nn_clf_test_recall:.4f}")

clf_results.append({'Model': 'Neural Network', 'Val_F1': nn_clf_val_f1, 'Test_F1': nn_clf_test_f1,
                    'Test_AUC': nn_clf_test_auc, 'Test_Precision': nn_clf_test_precision,
                    'Test_Recall': nn_clf_test_recall})

## 8. Head-to-Head Comparison

In [ ]:
reg_df = pd.DataFrame(reg_results)
clf_df = pd.DataFrame(clf_results)

print("=" * 80)
print("REGRESSION COMPARISON (Test Set)")
print("=" * 80)
print(f"\n{'Model':<18} {'Val R²':>10} {'Test R²':>10} {'Test RMSE':>12} {'Test MAE':>10} {'vs Ridge':>10}")
print("-" * 75)
ridge_r2 = reg_df[reg_df['Model'] == 'Ridge']['Test_R2'].values[0]
for _, row in reg_df.iterrows():
    delta = row['Test_R2'] - ridge_r2
    delta_str = f'{delta:+.4f}' if row['Model'] != 'Ridge' else '-'
    print(f"{row['Model']:<18} {row['Val_R2']:>10.4f} {row['Test_R2']:>10.4f} {row['Test_RMSE']:>12.4f} {row['Test_MAE']:>10.4f} {delta_str:>10}")

print("\n" + "=" * 80)
print("CLASSIFICATION COMPARISON (Test Set)")
print("=" * 80)
print(f"\n{'Model':<18} {'Val F1':>10} {'Test F1':>10} {'Test AUC':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 75)
for _, row in clf_df.iterrows():
    print(f"{row['Model']:<18} {row['Val_F1']:>10.4f} {row['Test_F1']:>10.4f} {row['Test_AUC']:>10.4f} {row['Test_Precision']:>10.4f} {row['Test_Recall']:>10.4f}")

In [ ]:
# Side-by-side comparison bar charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regression: R²
ax = axes[0]
models_reg = reg_df['Model'].values
r2_vals = reg_df['Test_R2'].values
colors_reg = ['steelblue', '#27ae60', '#9b59b6']
bars = ax.bar(models_reg, r2_vals, color=colors_reg, alpha=0.8)
for bar, val in zip(bars, r2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel(r'Test $R^2$')
ax.set_title('Regression: Test R² Comparison')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(r2_vals) * 1.15)

# Classification: F1
ax = axes[1]
models_clf = clf_df['Model'].values
f1_vals = clf_df['Test_F1'].values
colors_clf = ['steelblue', '#27ae60', '#e67e22', '#9b59b6']
bars = ax.bar(models_clf, f1_vals, color=colors_clf[:len(models_clf)], alpha=0.8)
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Test F1')
ax.set_title('Classification: Test F1 Comparison')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(f1_vals) * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter: Ridge, RF, NN
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scatter_models = [
    ('Ridge', ridge_test_pred, ridge_test_r2),
    ('Random Forest', rf_test_pred, rf_test_r2),
    ('Neural Network', nn_test_pred, nn_test_r2),
]

for ax, (name, preds, r2) in zip(axes, scatter_models):
    ax.scatter(y_test_reg, preds, alpha=0.4, s=20)
    ax.plot([0, 0.6], [0, 0.6], 'r--', label='Perfect prediction')
    ax.set_xlabel('Actual Delay Rate')
    ax.set_ylabel('Predicted Delay Rate')
    ax.set_title(f'{name} (R² = {r2:.3f})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 0.6)
    ax.set_ylim(0, 0.6)

plt.tight_layout()
plt.show()

## 9. Route-Level Comparison

In [ ]:
# Compute per-route R² for all three regression models
df_test = df_clean[test_mask].copy()
df_test['ridge_pred'] = ridge.predict(X_test_scaled)
df_test['rf_pred'] = rf_reg.predict(X_test)
df_test['nn_pred'] = nn_test_pred

print("Route-Level Regression Performance (Test Set):")
print("=" * 100)
print(f"{'Route':<22} {'Ridge R²':>10} {'RF R²':>10} {'NN R²':>10} {'NN vs Ridge':>12} {'NN vs RF':>10}")
print("-" * 100)

route_results = []
for route in sorted(df_test['route'].unique()):
    route_data = df_test[df_test['route'] == route]
    ridge_r2_route = r2_score(route_data['delay_rate'], route_data['ridge_pred'])
    rf_r2_route = r2_score(route_data['delay_rate'], route_data['rf_pred'])
    nn_r2_route = r2_score(route_data['delay_rate'], route_data['nn_pred'])
    
    delta_ridge = nn_r2_route - ridge_r2_route
    delta_rf = nn_r2_route - rf_r2_route
    
    print(f"{route:<22} {ridge_r2_route:>10.4f} {rf_r2_route:>10.4f} {nn_r2_route:>10.4f} {delta_ridge:>+12.4f} {delta_rf:>+10.4f}")
    route_results.append({
        'Route': route, 'Ridge_R2': ridge_r2_route, 'RF_R2': rf_r2_route,
        'NN_R2': nn_r2_route, 'n': len(route_data)
    })

print("-" * 100)
print(f"{'OVERALL':<22} {ridge_test_r2:>10.4f} {rf_test_r2:>10.4f} {nn_test_r2:>10.4f} "
      f"{nn_test_r2 - ridge_test_r2:>+12.4f} {nn_test_r2 - rf_test_r2:>+10.4f}")

route_compare_df = pd.DataFrame(route_results)

# Summary
nn_beats_ridge = (route_compare_df['NN_R2'] > route_compare_df['Ridge_R2']).sum()
nn_beats_rf = (route_compare_df['NN_R2'] > route_compare_df['RF_R2']).sum()
n_routes = len(route_compare_df)
print(f"\nNN outperforms Ridge on {nn_beats_ridge}/{n_routes} routes")
print(f"NN outperforms RF on {nn_beats_rf}/{n_routes} routes")

In [ ]:
# Route-level grouped bar chart
fig, ax = plt.subplots(figsize=(16, 6))

routes = route_compare_df['Route'].values
x = np.arange(len(routes))
width = 0.25

ax.bar(x - width, route_compare_df['Ridge_R2'], width, label='Ridge', color='steelblue', alpha=0.8)
ax.bar(x, route_compare_df['RF_R2'], width, label='Random Forest', color='#27ae60', alpha=0.8)
ax.bar(x + width, route_compare_df['NN_R2'], width, label='Neural Network', color='#9b59b6', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([r.replace('_', '\u2192') for r in routes], rotation=45, ha='right')
ax.set_ylabel(r'Test $R^2$')
ax.set_title('Route-Level R²: Ridge vs Random Forest vs Neural Network')
ax.axhline(0, color='black', linewidth=0.5)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Conclusions

In [ ]:
print("=" * 80)
print("NEURAL NETWORK COMPARISON: CONCLUSIONS")
print("=" * 80)

# Find best existing models
best_reg_existing = reg_df[reg_df['Model'] != 'Neural Network'].sort_values('Test_R2', ascending=False).iloc[0]
best_clf_existing = clf_df[clf_df['Model'] != 'Neural Network'].sort_values('Test_F1', ascending=False).iloc[0]

nn_reg_row = reg_df[reg_df['Model'] == 'Neural Network'].iloc[0]
nn_clf_row = clf_df[clf_df['Model'] == 'Neural Network'].iloc[0]

delta_r2 = nn_reg_row['Test_R2'] - best_reg_existing['Test_R2']
delta_f1 = nn_clf_row['Test_F1'] - best_clf_existing['Test_F1']

print(f"""
1. REGRESSION PERFORMANCE:
   Neural Network:  R² = {nn_reg_row['Test_R2']:.4f}, RMSE = {nn_reg_row['Test_RMSE']:.4f}
   Best existing:   R² = {best_reg_existing['Test_R2']:.4f} ({best_reg_existing['Model']}), RMSE = {best_reg_existing['Test_RMSE']:.4f}
   Difference:      \u0394R² = {delta_r2:+.4f}

2. CLASSIFICATION PERFORMANCE:
   Neural Network:  F1 = {nn_clf_row['Test_F1']:.4f}, AUC = {nn_clf_row['Test_AUC']:.4f}
   Best existing:   F1 = {best_clf_existing['Test_F1']:.4f} ({best_clf_existing['Model']}), AUC = {best_clf_existing['Test_AUC']:.4f}
   Difference:      \u0394F1 = {delta_f1:+.4f}

3. ROUTE-LEVEL ANALYSIS:
   NN outperforms Ridge on {nn_beats_ridge}/{n_routes} routes
   NN outperforms RF on    {nn_beats_rf}/{n_routes} routes

4. TRAINING CHARACTERISTICS:
   Regression converged at epoch {best_epoch_reg} (of 500 max)
   Classification converged at epoch {best_epoch_clf} (of 500 max)
   Model parameters: {nn_reg.count_params():,} (vs {X_train_scaled.shape[0]:,} training samples)
""")


### Observations

- The neural network's performance is comparable to existing models () 
- Given the added complexity, training instability, and reduced interpretability, the simpler models are still preferred overall
   - Ridge for regression, XGBoost for classification
- The results again confirm that the performance ceiling is determined by data availability and the inherent unpredictability of monthly delay rates, not by model complexity.

## 11. Next Step

The next notebook will explore adding "load factor" (passenger-to-capacity ratio) as a feature in the prediction model.